# CSI Data Collector — v2
**Wi-Fi Sensing with ESP32-S3 | Final Course Project — UFMG**

This version generates, in addition to the raw `.csv` data file, a `.json` metadata file per session.  
Intended usage flow:

1. **Cell 1 — Imports & utilities** *(run once per kernel session)*
2. **Cell 2 — Fixed setup metadata** *(fill in once per collection day)*
3. **Cell 3 — Session parameters** *(fill in before each individual session)*
4. **Cell 4 — Data collection** *(run to start recording)*
5. **Cell 5 — Wrap-up & JSON export** *(run immediately after the recording stops)*


---
## Cell 1 — Imports & utilities
Run once at the beginning of the work session.

In [ ]:
import serial
import time
import json
import os
from datetime import datetime, timezone
from pathlib import Path

# -------------------------------------------------------------------
# Hardware constants (unchanged between sessions)
# -------------------------------------------------------------------
PORT  = "/dev/ttyUSB0"   # serial port of the RX node
BAUD  = 921600           # must match the csi_recv firmware baud rate

HEADER = (
    "timestamp_host,"
    "type,id,mac,rssi,rate,sig_mode,mcs,bandwidth,smoothing,"
    "not_sounding,aggregation,stbc,fec_coding,sgi,noise_floor,"
    "ampdu_cnt,channel,secondary_channel,local_timestamp,ant,"
    "sig_len,rx_state,len,first_word,data\n"
)

FOLDER_NAME = "pilot"
DATA_DIR = Path(FOLDER_NAME) / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------------------
# Utility functions
# -------------------------------------------------------------------
def _ts_iso() -> str:
    """Returns a local ISO 8601 timestamp with UTC offset."""
    return datetime.now().astimezone().isoformat(timespec='seconds')

def _ts_host() -> str:
    """Host timestamp used to prefix each CSV row."""
    return datetime.now().strftime("%Y%m%dT%H%M%S.%f")

def _build_filename(session_id: str, label: str, dt: datetime) -> str:
    return f"session_{session_id}_{label}_{dt.strftime('%Y%m%d_%H%M')}"

print("Imports ready.")


---
## Cell 2 — Fixed setup metadata
Fill in **once per collection day** (or whenever the physical setup changes).  
These values will be embedded in every JSON file generated during this work session.

In [ ]:
# ===================================================================
#  FILL IN MANUALLY
# ===================================================================

SETUP_META = {

    # --- Physical environment -------------------------------------
    "room": {
        "description"       : "Residential bedroom, brick walls, concrete slab ceiling",
        "dimensions_m"      : {"east_west": 3.40, "north_south": 3.45, "ceiling_height": 2.85},
        "door_material"     : "plywood",
        "window_material"   : "iron and glass",
        "notable_objects"   : ["MDF wardrobe", "mirror", "bed", "desk", "book niches x2"]
    },

    # --- Node positioning -----------------------------------------
    "nodes": {
        "tx": {
            "role"                     : "STA (ICMP transmitter)",
            "device"                   : "ESP32-S3-DevKitC-1",
            "tripod_height_m"          : 1.20,
            "pos_x_from_west_wall_m"   : 1.45,
            "pos_y_from_north_wall_m"  : 3.08,
            "mac"                      : "1a:00:00:00:00:00"   # fill in
        },
        "rx": {
            "role"                     : "AP (CSI receiver)",
            "device"                   : "ESP32-S3-DevKitC-1",
            "tripod_height_m"          : 1.20,
            "pos_x_from_west_wall_m"   : 1.45,
            "pos_y_from_north_wall_m"  : 1.14,
            "mac"                      : "1c:db:d4:9d:93:dc",  # fill in
            "serial_port"              : PORT,
            "baud_rate"                : BAUD
        },
        "tx_rx_los_distance_m"         : 2.00,
        "tx_rx_axis"                   : "parallel to north-south axis"
    },

    # --- Firmware / Wi-Fi parameters -----------------------------
    "wifi": {
        "ssid"              : "CSI_AP",              # confirm
        "channel"           : 11,                     # confirm
        "bandwidth_mhz"     : 40,
        "icmp_rate_hz"      : 30,                    # ping rate at TX
        "protocol"          : "ESP-NOW HT40"
    },

    # --- Collection protocol -------------------------------------
    "protocol": {
        "stabilization_s"   : 60,    # seconds of link stabilization before condition starts
        "active_window_s"   : 600,   # labeled collection window (10 min)
        "buffer_end_s"      : 30,    # safety buffer before stopping recording
        "subject_position"  : {"x_from_west_m": 1.45, "y_from_north_m": 2.11},
        "subject_orientation": "facing north (towards RX node)"
    },

    # --- Label descriptions --------------------------------------
    "labels_description": {
        "empty"             : "No human presence; operator outside the room with door closed.",
        "occupied_still"    : "Subject standing at center mark, motionless.",
        "occupied_moving"   : "Subject standing at center mark, slow movements within +/-0.30 m."
    }
}

print("Setup metadata loaded.")
print(json.dumps(SETUP_META, indent=2, ensure_ascii=False))


## Cell 3 — Session Configuration

Edit **Cell 3** before each session:

- `SESSION_ID` — single letter identifying this session within the day (A, B, C, ...).
- `LABEL` — one of `"empty"`, `"occupied_still"`, or `"occupied_moving"`.
- `DURATION` — total recording duration in seconds.
- `START_DELAY` — countdown before the serial port opens (use this time to leave or enter the room).
- `ENV_COND` — environmental conditions at the time of the session (optional but recommended).

The CSV and JSON filenames are generated automatically from these values.


In [ ]:
# ===================================================================
#  FILL IN MANUALLY BEFORE EACH SESSION
# ===================================================================

SESSION_ID   = "E"               # sequential per day: A, B, C ...
LABEL        = "occupied_moving"  # "empty" | "occupied_still" | "occupied_moving"
DURATION     = 690               # total recording duration in seconds (>= stab + active + buffer)
START_DELAY  = 15                # countdown in seconds before recording starts

STABILIZATION_S  = SETUP_META["protocol"]["stabilization_s"]   # 60
ACTIVE_WINDOW_S  = SETUP_META["protocol"]["active_window_s"]    # 600

# Environmental conditions at the time of the session
ENV_COND = {
    "temperature_C"          : None,   # fill in (e.g. 24)
    "visible_wifi_networks"  : None,   # fill in (e.g. 3)
    "avg_rssi_dbm"           : None,   # fill in after session (e.g. -45)
    "observed_interferences" : None,   # fill in (e.g. "none" or "microwave in adjacent room")
    "door_state"             : "closed",  # "closed" | "open"
    "window_state"           : "closed",  # "closed" | "open"
    "free_notes"             : ""      # free-text field
}

# Basic validations
_valid_labels = {"empty", "occupied_still", "occupied_moving"}
assert LABEL in _valid_labels, f"Invalid LABEL. Use one of: {_valid_labels}"
assert SESSION_ID.isalpha() and len(SESSION_ID) <= 3, "SESSION_ID must be letter(s): A, B, C ..."

_dt_session   = datetime.now()
_base_name    = _build_filename(SESSION_ID, LABEL, _dt_session)
CSV_FILE      = str(DATA_DIR / f"{_base_name}.csv")
JSON_FILE     = str(DATA_DIR / f"{_base_name}_meta.json")

print(f"Session ID   : {SESSION_ID}")
print(f"Label        : {LABEL}")
print(f"Duration     : {DURATION} s")
print(f"CSV file     : {CSV_FILE}")
print(f"JSON file    : {JSON_FILE}")
print(f"\nDouble-check the label and session ID before running Cell 4.")


## Cell 5 — Timestamp Logging and JSON Export

> **Write down the exact time** shown on the cell output right after Cell 4 finishes.
> Use those values for `T1` and `T2` below (or leave `None` to accept the automatic values).

- `T1` — start of the labeled activity window (`HH:MM:SS`).
- `T2` — end of the labeled activity window (`HH:MM:SS`).
- `INVALIDATION_REASON` — if this session should be discarded, describe why; otherwise leave it empty.


In [ ]:
# ---
# Main recording loop.
# Opens the serial port, writes the CSV header, then captures frames
# until DURATION seconds have elapsed or the user interrupts manually.
# ---

import serial
import time
from datetime import datetime

_t1_auto = None   # automatic timestamps set during recording
_t2_auto = None

print(f"Starting in {START_DELAY} s — leave the room / take your position now...")
time.sleep(START_DELAY)

print(f"\nRECORDING: {CSV_FILE}")
print("Press [Interrupt] to stop early.\n")

samples = 0
_t_start = time.time()

with serial.Serial(PORT, BAUD, timeout=1) as ser, open(CSV_FILE, "w") as f:
    f.write(HEADER)
    _t1_auto = datetime.now()

    while (time.time() - _t_start) < DURATION:
        line = ser.readline()
        if line and line.startswith(b"CSI_DATA"):
            ts = _ts_host()
            f.write(f"{ts},{line.decode('utf-8', errors='replace').strip()}\n")
            samples += 1

_t2_auto = datetime.now()

print(f"\nRecording complete: {samples} samples written to '{CSV_FILE}'")
print(f"\nProceed to Cell 5 to log the ground-truth timestamps and export the JSON.")
print(f"\nAutomatic t1: {_t1_auto.strftime('%H:%M:%S')}")
print(f"Automatic t2: {_t2_auto.strftime('%H:%M:%S')}")
print(f"   Use these values for T1 and T2 in Cell 5.")


---
### Calculating `avg_rssi_dbm`



In [ ]:
import pandas as pd

df = pd.read_csv(CSV_FILE)
rssi_mean = round(df["rssi"].mean(), 2)
print(f"avg_rssi_dbm → {rssi_mean}")

---
## Cell 5 — Wrap-up, ground truth & JSON export

Fill in `T1` and `T2` with the times you noted during the session:  
- **`T1`** → moment the condition started (left the room for *empty*; entered and took position for *occupied*)
- **`T2`** → moment the condition ended

Accepted format: `"HH:MM:SS"` — the date is automatically inferred from `t0`.

In [ ]:
from datetime import datetime

# ===================================================================
#  FILL IN MANUALLY (or leave None to use automatic values)
# ===================================================================

T1 = None   # leave None to use the automatic value computed in Cell 4
T2 = None   # leave None to use the automatic value computed in Cell 4

INVALIDATION_REASON = ""   # leave empty if the session is valid

# Resolve automatic values if not manually set:
_today = _dt_session.strftime("%Y-%m-%d")
if T1 is None and _t1_auto is not None:
    T1 = datetime.fromisoformat(f"{_today}T{_t1_auto.strftime('%H:%M:%S')}")
    print(f"T1 (automatic): {T1}")
if T2 is None and _t2_auto is not None:
    T2 = datetime.fromisoformat(f"{_today}T{_t2_auto.strftime('%H:%M:%S')}")
    print(f"T2 (automatic): {T2}")

assert T1 and T2, "T1 and T2 must be set before exporting the JSON."
assert T1 < T2,   "T1 must be earlier than T2."

active_window = (T2 - T1).total_seconds()

# ---
# Build and export the metadata JSON
# ---
meta = {
    "session_id"           : SESSION_ID,
    "label"                : LABEL,
    "is_valid"             : not bool(INVALIDATION_REASON),
    "invalidation_reason"  : INVALIDATION_REASON or None,
    "created_at"           : _ts_iso(),
    "csv_file"             : CSV_FILE,
    "json_file"            : JSON_FILE,
    "t1_iso"               : T1.isoformat(timespec='seconds'),
    "t2_iso"               : T2.isoformat(timespec='seconds'),
    "active_window_s"      : active_window,
    "total_samples"        : samples,
    "setup"                : SETUP_META,
    "env_conditions"       : ENV_COND
}

with open(JSON_FILE, "w") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print(f"JSON exported: {JSON_FILE}")

if INVALIDATION_REASON:
    print(f"\n[INVALID] Session marked as invalid: {INVALIDATION_REASON}")

print(f"\nSession summary:")
print(f"   Session ID       : {SESSION_ID}")
print(f"   Label            : {LABEL}")
print(f"   Samples recorded : {samples}")
print(f"   Active window    : {active_window} s  (t1 -> t2)")
print(f"   t1               : {T1}")
print(f"   t2               : {T2}")
print(f"\nFiles ready for the dataset:")
print(f"   - {CSV_FILE}")
print(f"   - {JSON_FILE}")


---
## Cell 6 — Inspect the generated JSON *(optional)*
Useful for verifying the metadata file contents before archiving.

In [ ]:
with open(JSON_FILE, encoding="utf-8") as jf:
    _contents = json.load(jf)

print(json.dumps(_contents, indent=2, ensure_ascii=False))


---
## Cell 7 — List sessions of the day *(optional)*
Lists all `csv + json` pairs found in the current working directory.

In [ ]:
"""
Quick audit of the data/ directory: lists every session CSV/JSON pair
and flags missing metadata files.
"""
import json
from pathlib import Path

data_dir = DATA_DIR

csvs  = {p.stem: p for p in data_dir.glob("*.csv")}
jsons = {p.stem.removesuffix("_meta"): p for p in data_dir.glob("*_meta.json")}

print(f"Data directory: {data_dir.resolve()}")
print(f"{'Session':<40} {'CSV':>5}  {'JSON':>5}")
print("-" * 55)

all_keys = sorted(set(csvs) | set(jsons))
for key in all_keys:
    has_csv  = "yes" if key in csvs  else "no"
    has_json = "yes" if key in jsons else "no"
    print(f"{key:<40} {has_csv:>5}  {has_json:>5}")

print(f"\nTotal: {len(csvs)} CSV file(s), {len(jsons)} JSON file(s).")
